# 18 Master orchestration

This notebook inspects the current pipeline state, builds a recommended run plan, and exports orchestration tables.
It is **non-destructive** and is meant to guide or coordinate the next notebook to run for a selected batch/branch.


In [ ]:
from pathlib import Path
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from src.orchestrator import (
    build_stage_status,
    build_run_plan,
    export_orchestration_tables,
    infer_next_notebook,
    summarize_outputs,
)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
INCLUDE_OCR = True
INCLUDE_APPLY = False
INCLUDE_ROLLBACK = False
INCLUDE_QUALITY = True
SHOW_RECENT_OUTPUTS = True


In [ ]:
status_df = build_stage_status(OUTPUT_DIR)
display(status_df)


In [ ]:
run_plan_df = build_run_plan(
    OUTPUT_DIR,
    include_ocr=INCLUDE_OCR,
    include_apply=INCLUDE_APPLY,
    include_rollback=INCLUDE_ROLLBACK,
    include_quality=INCLUDE_QUALITY,
)
display(run_plan_df)


In [ ]:
next_nb = infer_next_notebook(run_plan_df)
if next_nb is None:
    print('All selected stages already have outputs.')
else:
    print('Recommended next notebook:', next_nb)


In [ ]:
if SHOW_RECENT_OUTPUTS:
    recent_outputs = summarize_outputs(OUTPUT_DIR)
    display(recent_outputs.head(50))


In [ ]:
status_path, plan_path = export_orchestration_tables(status_df, run_plan_df, OUTPUT_DIR)
print('Wrote:', status_path)
print('Wrote:', plan_path)


## Typical use

- Keep `INCLUDE_APPLY = False` while you are still validating and refining.
- Turn `INCLUDE_OCR = True` only when you want scanned/image files in the path.
- After accepted feedback reruns, use this notebook to see whether the next notebook should be the normal promotion path (`14_feedback_to_execution.ipynb`) or the OCR promotion path (`17_ocr_feedback_to_execution.ipynb`).
